# Volatility Arbitrage Strategy

This notebook demonstrates **implied vs realized volatility arbitrage** - an options-like strategy for asset portfolios.

**Core Principle**: When two assets are highly correlated, their IV/RV ratios should converge. Divergence creates arbitrage opportunities.

**Strategy Components**:
1. **IV/RV Ratio Calculation** - Implied vs realized volatility spreads
2. **Correlation Filtering** - Only trade pairs with ρ > 0.85
3. **Dispersion Trading** - Exploit ratio divergence in correlated pairs
4. **Risk-Neutral Pricing** - Delta-neutral position construction
5. **Greeks Analysis** - Vega exposure and gamma hedging
6. **VIX Regime Analysis** - Performance across volatility regimes

**Paper References**:
- Moghaddam 2018 (arXiv:1810.07735) - Volatility Dispersion Trading
- Grinold-Kahn 1999 - Active Portfolio Management
- Rockafellar & Uryasev 2000 - CVaR optimization

**Unique Angle**: Apply options volatility arbitrage concepts to equity portfolios using correlation structures.

---

## Setup

In [ ]:
# Add parent directory to pathimport sysfrom pathlib import Pathsys.path.insert(0, str(Path.cwd().parent))# Standard importsimport numpy as npimport pandas as pdimport polars as plfrom datetime import date, timedeltafrom typing import List, Dict, Tuple# Visualizationimport matplotlib.pyplot as pltimport seaborn as snssns.set_style('whitegrid')plt.rcParams['figure.figsize'] = (14, 6)# ARBS framework imports (CORRECTED PATHS)from Signals.CorrelationVolatilitySignal import CorrelationVolatilitySignalfrom Risk.Covariance.LedoitWolfShrinkage import LedoitWolfShrinkagefrom Signals.AlphaGenerator import AlphaGeneratorfrom Optimizer.MeanVarianceOptimizer import MeanVarianceOptimizerfrom Risk.Volatility.RealizedVolatility import RealizedVolatilityprint("✓ Setup complete!")

---

## Part 1: Generate Mock Market Data

We'll create synthetic data for SPDR sector ETFs with realistic correlation structures.

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# SPDR Sector ETFs
tickers = ['XLK', 'XLF', 'XLE', 'XLV', 'XLI', 'XLP', 'XLY', 'XLU', 'XLB', 'XLRE', 'XLC']

# Sector classifications for correlation structure
sector_groups = {
    'Tech': ['XLK', 'XLC'],  # Technology, Communication - high correlation
    'Financials': ['XLF', 'XLRE'],  # Financials, Real Estate - correlated
    'Energy': ['XLE'],  # Energy - low correlation
    'Healthcare': ['XLV'],
    'Industrials': ['XLI'],
    'Consumer': ['XLP', 'XLY'],  # Staples and Discretionary
    'Utilities': ['XLU'],
    'Materials': ['XLB']
}

# Generate 2 years of daily data (504 trading days)
n_days = 504
start_date = date(2023, 1, 1)
dates = [start_date + timedelta(days=i) for i in range(n_days)]

print(f"Generating {n_days} days of data for {len(tickers)} ETFs...")
print(f"Period: {dates[0]} to {dates[-1]}")

In [ ]:
def generate_correlated_returns(
    tickers: List[str],
    sector_groups: Dict[str, List[str]],
    n_days: int,
    base_vol: float = 0.01
) -> Dict[str, np.ndarray]:
    """
    Generate returns with sector-level correlation structure.
    
    High correlation within sectors (0.7-0.9)
    Moderate correlation across sectors (0.3-0.5)
    """
    # Create market factor
    market_returns = np.random.normal(0.0005, base_vol, n_days)
    
    # Create sector factors
    sector_factors = {}
    for sector in sector_groups.keys():
        sector_factors[sector] = np.random.normal(0, base_vol * 0.5, n_days)
    
    # Generate asset returns with correlation
    returns_dict = {}
    
    for ticker in tickers:
        # Find sector
        ticker_sector = None
        for sector, members in sector_groups.items():
            if ticker in members:
                ticker_sector = sector
                break
        
        if ticker_sector is None:
            ticker_sector = 'Other'
            sector_factors[ticker_sector] = np.random.normal(0, base_vol * 0.5, n_days)
        
        # Determine beta to market and sector
        if ticker in ['XLK', 'XLC']:  # Tech - high beta
            market_beta = 1.2
            sector_beta = 0.8
            idio_vol = base_vol * 0.4
        elif ticker in ['XLF', 'XLRE']:  # Financials - high correlation
            market_beta = 1.1
            sector_beta = 0.9
            idio_vol = base_vol * 0.3
        elif ticker == 'XLE':  # Energy - low correlation
            market_beta = 0.5
            sector_beta = 0.3
            idio_vol = base_vol * 1.5
        else:  # Others - moderate
            market_beta = 0.9
            sector_beta = 0.7
            idio_vol = base_vol * 0.5
        
        # Asset return = market + sector + idiosyncratic
        idiosyncratic = np.random.normal(0, idio_vol, n_days)
        asset_returns = (
            market_beta * market_returns +
            sector_beta * sector_factors[ticker_sector] +
            idiosyncratic
        )
        
        returns_dict[ticker] = asset_returns
    
    return returns_dict

# Generate returns
returns_dict = generate_correlated_returns(tickers, sector_groups, n_days)

# Create returns DataFrame (long format)
returns_data = []
for ticker in tickers:
    for i, d in enumerate(dates):
        returns_data.append({
            'date': d,
            'ticker': ticker,
            'return': returns_dict[ticker][i]
        })

returns_df = pl.DataFrame(returns_data)

print(f"✓ Generated {len(returns_data)} return observations")
print(f"\nSample (first 5 rows):")
print(returns_df.head(5))

---

## Part 2: Calculate Correlation Matrix

Identify highly correlated pairs (ρ > 0.85) for volatility dispersion trading.

In [ ]:
# Pivot returns to wide format for correlation calculation
returns_wide = returns_df.pivot(
    index='date',
    columns='ticker',
    values='return'
).to_pandas()

# Calculate correlation matrix
corr_matrix = returns_wide.corr()

print(f"✓ Calculated correlation matrix ({corr_matrix.shape[0]}x{corr_matrix.shape[1]})")
print(f"\nSample correlations (first 3x3):")
print(corr_matrix.iloc[:3, :3])

In [ ]:
# Visualize correlation matrix
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            vmin=-0.5, vmax=1.0, square=True, cbar_kws={'label': 'Correlation'})
plt.title('SPDR Sector ETF Return Correlations', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📊 Key Observations:")
print("Notice high correlations within sector groups:")
print("  • XLK-XLC (Technology-Communication): High correlation expected")
print("  • XLF-XLRE (Financials-Real Estate): Correlated due to interest rate sensitivity")
print("  • XLE (Energy): Lower correlation with other sectors")

In [ ]:
# Find high-correlation pairs (ρ > 0.85)
def find_high_correlation_pairs(
    corr_matrix: pd.DataFrame,
    min_correlation: float = 0.85
) -> List[Tuple[str, str, float]]:
    """Find asset pairs with correlation above threshold."""
    pairs = []
    
    for i, ticker1 in enumerate(corr_matrix.index):
        for j, ticker2 in enumerate(corr_matrix.columns):
            if i < j:  # Avoid duplicates and self-correlation
                corr = corr_matrix.loc[ticker1, ticker2]
                if corr >= min_correlation:
                    pairs.append((ticker1, ticker2, corr))
    
    # Sort by correlation (descending)
    pairs.sort(key=lambda x: x[2], reverse=True)
    
    return pairs

high_corr_pairs = find_high_correlation_pairs(corr_matrix, min_correlation=0.85)

print(f"✓ Found {len(high_corr_pairs)} pairs with correlation >= 0.85:")
print("\n" + "=" * 50)
print(f"{'Pair':<15} {'Correlation':<15} {'Sector Affinity'}")
print("=" * 50)

for ticker1, ticker2, corr in high_corr_pairs:
    # Find sector groups
    sectors1 = [s for s, members in sector_groups.items() if ticker1 in members]
    sectors2 = [s for s, members in sector_groups.items() if ticker2 in members]
    sector1 = sectors1[0] if sectors1 else 'Other'
    sector2 = sectors2[0] if sectors2 else 'Other'
    
    affinity = "Same" if sector1 == sector2 else "Different"
    
    print(f"{ticker1}-{ticker2:<10} {corr:>8.3f}        {affinity}")

print("\n💡 Trading Rationale:")
print("When two assets are highly correlated (ρ > 0.85), their volatilities should also")
print("behave similarly. If IV/RV ratios diverge significantly, it creates an arbitrage")
print("opportunity - we sell expensive volatility and buy cheap volatility.")

In [ ]:
# Define helper class for volatility ratio calculation
class VolatilityRatioCalculator:
    """
    Calculate IV/RV ratios for volatility arbitrage.
    
    This is a simplified mock implementation for educational purposes.
    In production, use real options IV data from:
    - CBOE (VIX, VVIX)
    - IVolatility API
    - Bloomberg IVOL function
    """
    
    def __init__(self, lookback: int = 30, annualization: int = 252):
        self.lookback = lookback
        self.annualization = annualization
    
    def calculate_realized_volatility(self, returns_df: pl.DataFrame) -> pl.DataFrame:
        """Calculate rolling realized volatility for each ticker."""
        
        # Pivot to wide format
        returns_wide = returns_df.pivot(
            index='date',
            columns='ticker',
            values='return'
        ).to_pandas()
        
        # Calculate rolling volatility for each ticker
        rv_data = []
        for ticker in returns_wide.columns:
            if ticker == 'date':
                continue
            
            returns_series = returns_wide[ticker]
            
            # Rolling standard deviation
            rolling_std = returns_series.rolling(window=self.lookback, min_periods=self.lookback).std()
            
            # Annualize
            rv = rolling_std * np.sqrt(self.annualization)
            
            # Take latest value
            latest_rv = rv.iloc[-1] if not np.isnan(rv.iloc[-1]) else 0.15  # Default to 15%
            
            rv_data.append({
                'ticker': ticker,
                'RV': latest_rv
            })
        
        return pl.DataFrame(rv_data)
    
    def calculate_ratios(self, 
                        returns_df: pl.DataFrame,
                        implied_vols: Dict[str, float]) -> pl.DataFrame:
        """Calculate IV/RV ratios."""
        
        # Get RV
        rv_df = self.calculate_realized_volatility(returns_df)
        
        # Merge with IV
        ratio_data = []
        for row in rv_df.iter_rows(named=True):
            ticker = row['ticker']
            rv = row['RV']
            iv = implied_vols.get(ticker, rv * 1.15)  # Default: 15% premium
            
            ratio_data.append({
                'ticker': ticker,
                'RV': rv,
                'IV': iv,
                'IV_RV_ratio': iv / rv if rv > 0 else 1.0
            })
        
        return pl.DataFrame(ratio_data)

print("✓ VolatilityRatioCalculator defined")

---

## Part 3: Calculate IV/RV Ratios

Calculate implied vs realized volatility ratios using rolling windows.

In [ ]:
# Initialize volatility ratio calculator
vol_calc = VolatilityRatioCalculator(
    lookback=30,  # 30-day rolling window for RV
    annualization=252  # Daily data
)

# Generate mock implied volatilities
def generate_mock_implied_vols(
    returns_df: pl.DataFrame,
    vol_calc: VolatilityRatioCalculator,
    iv_premium: float = 0.15
) -> Dict[str, float]:
    """
    Generate mock implied volatilities.
    
    Formula: IV = RV × (1.0 + premium + noise)
    
    In real markets:
    - IV > RV (volatility risk premium)
    - IV/RV ratio typically between 1.0 and 1.5
    - Spikes during market stress (VIX regime shifts)
    """
    # Calculate latest RV for each ticker
    rv_df = vol_calc.calculate_realized_volatility(returns_df)
    
    # Generate IV with premium and noise
    np.random.seed(123)  # Separate seed for IV
    implied_vols = {}
    
    for row in rv_df.iter_rows(named=True):
        ticker = row['ticker']
        rv = row['RV']
        
        # Add volatility risk premium + noise
        noise = np.random.normal(0, 0.10)  # 10% noise
        iv = rv * (1.0 + iv_premium + noise)
        
        # Ensure IV is positive and reasonable
        iv = max(iv, 0.01)
        iv = min(iv, rv * 2.0)  # Cap at 2x RV
        
        implied_vols[ticker] = iv
    
    return implied_vols

# Generate implied vols
implied_vols = generate_mock_implied_vols(returns_df, vol_calc, iv_premium=0.15)

print("✓ Generated mock implied volatilities (annualized):\n")
print("=" * 50)
print(f"{'Ticker':<10} {'IV (%)':<10} {'Expected RV Premium'}")
print("=" * 50)

for ticker in sorted(implied_vols.keys()):
    iv = implied_vols[ticker]
    print(f"{ticker:<10} {iv*100:>8.2f}%")

print("\n⚠️ Note: Using mock IV data. In production, use real options data:")
print("  • CBOE (e.g., VIX for SPX)")
print("  • IVolatility API")
print("  • Bloomberg IVOL function")
print("  • Options chain implied volatility")

In [ ]:
# Calculate IV/RV ratios using VolatilityRatioCalculator
vol_ratios = vol_calc.calculate_ratios(returns_df, implied_vols)

print("✓ Calculated IV/RV ratios:\n")
print(vol_ratios.sort('ticker'))

# Visualize IV vs RV
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Convert to pandas for plotting
vol_ratios_pd = vol_ratios.to_pandas()
vol_ratios_pd = vol_ratios_pd.sort_values('ticker')

# Bar chart: IV vs RV
x = np.arange(len(vol_ratios_pd))
width = 0.35

axes[0].bar(x - width/2, vol_ratios_pd['RV'] * 100, width, label='Realized Vol', alpha=0.7, color='steelblue')
axes[0].bar(x + width/2, vol_ratios_pd['IV'] * 100, width, label='Implied Vol', alpha=0.7, color='orange')
axes[0].set_xlabel('ETF', fontsize=10)
axes[0].set_ylabel('Volatility (%)', fontsize=10)
axes[0].set_title('Implied vs Realized Volatility', fontsize=12, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(vol_ratios_pd['ticker'], rotation=45)
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Bar chart: IV/RV ratios
axes[1].bar(vol_ratios_pd['ticker'], vol_ratios_pd['IV_RV_ratio'], alpha=0.7, color='green')
axes[1].axhline(y=1.0, color='red', linestyle='--', linewidth=2, label='Parity (IV=RV)')
axes[1].set_xlabel('ETF', fontsize=10)
axes[1].set_ylabel('IV/RV Ratio', fontsize=10)
axes[1].set_title('IV/RV Ratio by ETF', fontsize=12, fontweight='bold')
axes[1].set_xticklabels(vol_ratios_pd['ticker'], rotation=45)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n💡 Interpretation:")
print("  • IV > RV (ratio > 1.0): Market expects higher future volatility")
print("  • IV < RV (ratio < 1.0): Market expects lower future volatility (rare)")
print("  • Large deviations from 1.0 create arbitrage opportunities")

---

## Part 4: Generate Volatility Dispersion Signals

Use CorrelationVolatilitySignal to identify arbitrage opportunities.

In [ ]:
# Initialize signal generator
vol_signal = CorrelationVolatilitySignal(
    min_correlation=0.85,  # Only trade highly correlated pairs
    lookback=60,  # 60-day window for z-score calculation
    z_threshold=1.5,  # Signal when |z_score| > 1.5
    standardize=True,
    track_history=True
)

# Extract pairs (without correlation value)
pairs = [(ticker1, ticker2) for ticker1, ticker2, _ in high_corr_pairs]

# Calculate signals
signals_df = vol_signal.calculate_batch_detailed(
    returns=returns_df,
    vol_ratios=vol_ratios,
    pairs=pairs,
    as_of=dates[-1]  # Latest date
)

print(f"✓ Generated volatility dispersion signals")
print(f"\nTotal pairs analyzed: {len(pairs)}")
print(f"Signals above threshold: {signals_df.height}")

if signals_df.height > 0:
    print("\n" + "=" * 90)
    print("VOLATILITY DISPERSION SIGNALS")
    print("=" * 90)
    print(signals_df.sort('signal', descending=True))
    
    print("\n💡 Signal Interpretation:")
    print("  • spread: IV_RV_B - IV_RV_A (positive = B's ratio is higher)")
    print("  • z_score: Standardized spread (how many std devs from mean)")
    print("  • signal: |z_score| × correlation (signal strength)")
    print("  • direction:")
    print("      - sell_B_vol: B's volatility is expensive → sell B, buy A")
    print("      - sell_A_vol: A's volatility is expensive → sell A, buy B")
else:
    print("\n⚠️ No signals above threshold at current date")
    print("This can happen when:")
    print("  • IV/RV ratios are converged (no arbitrage opportunity)")
    print("  • Threshold is too high (try lowering z_threshold)")
    print("  • Mock data doesn't have enough divergence")

---

## Part 5: Risk-Neutral Pricing Framework

Construct delta-neutral positions to isolate volatility exposure.

In [ ]:
def calculate_delta_neutral_hedge(
    signal_row: Dict,
    notional: float = 100000
) -> Dict:
    """
    Calculate delta-neutral hedge ratios for volatility trade.
    
    Strategy:
    1. Long cheap volatility asset (weight = +1)
    2. Short expensive volatility asset (weight = -hedge_ratio)
    3. Adjust for correlation to maintain delta neutrality
    
    Args:
        signal_row: Signal data for a pair
        notional: Total position size
    
    Returns:
        Dict with position details
    """
    asset_A = signal_row['asset_A']
    asset_B = signal_row['asset_B']
    correlation = signal_row['correlation']
    direction = signal_row['direction']
    
    # Hedge ratio (adjust for correlation)
    # Higher correlation = closer to 1:1 hedge
    hedge_ratio = correlation
    
    if direction == 'sell_B_vol':
        # B is expensive, A is cheap
        # Long A, Short B
        position_A = notional / 2
        position_B = -notional / 2 * hedge_ratio
        long_asset = asset_A
        short_asset = asset_B
    else:
        # A is expensive, B is cheap
        # Long B, Short A
        position_B = notional / 2
        position_A = -notional / 2 * hedge_ratio
        long_asset = asset_B
        short_asset = asset_A
    
    return {
        'pair': signal_row['pair'],
        'long_asset': long_asset,
        'short_asset': short_asset,
        f'position_{asset_A}': position_A,
        f'position_{asset_B}': position_B,
        'hedge_ratio': hedge_ratio,
        'correlation': correlation,
        'net_delta': position_A + position_B  # Should be ~0 for delta-neutral
    }

# Calculate delta-neutral positions
if signals_df.height > 0:
    print("✓ Calculating delta-neutral hedge positions...\n")
    print("=" * 80)
    print("DELTA-NEUTRAL POSITIONS (Notional: $100,000 per pair)")
    print("=" * 80)
    
    for row in signals_df.iter_rows(named=True):
        position = calculate_delta_neutral_hedge(row, notional=100000)
        
        print(f"\nPair: {position['pair']}")
        print(f"  Long:  {position['long_asset']:<6} ${position[f'position_{position[\"long_asset\"]}']:>10,.0f}")
        print(f"  Short: {position['short_asset']:<6} ${position[f'position_{position[\"short_asset\"]}']:>10,.0f}")
        print(f"  Hedge Ratio: {position['hedge_ratio']:.3f}")
        print(f"  Net Delta: ${position['net_delta']:>10,.0f} (target: $0)")
    
    print("\n💡 Delta-Neutral Trading:")
    print("  • Positions are sized to cancel out directional market exposure")
    print("  • Profit comes from volatility convergence, not price movements")
    print("  • Correlation-adjusted hedge ratio ensures market neutrality")
    print("  • Daily rebalancing required to maintain delta neutrality")
else:
    print("⚠️ No positions to construct (no signals above threshold)")

---

## Part 6: Greeks Analysis (Vega Exposure)

Analyze sensitivity to volatility changes (vega) for each position.

In [ ]:
def calculate_vega_exposure(
    vol_ratios_df: pl.DataFrame,
    position_size: float,
    ticker: str
) -> float:
    """
    Calculate vega exposure (P&L change per 1% vol change).
    
    Simplified calculation:
    Vega ≈ position_size × RV × sensitivity
    
    In real options:
    Vega = ∂V/∂σ (option value change per vol change)
    """
    # Get realized volatility for ticker
    ticker_data = vol_ratios_df.filter(pl.col('ticker') == ticker)
    
    if ticker_data.height == 0:
        return 0.0
    
    rv = ticker_data['RV'][0]
    
    # Vega = position × volatility
    # Sensitivity: 1% vol change ≈ 0.01 × position × vol
    vega = position_size * rv * 0.01
    
    return vega

if signals_df.height > 0:
    print("✓ Calculating vega exposure (volatility Greeks)...\n")
    print("=" * 80)
    print("VEGA EXPOSURE (P&L per 1% volatility change)")
    print("=" * 80)
    
    total_long_vega = 0.0
    total_short_vega = 0.0
    
    for row in signals_df.iter_rows(named=True):
        asset_A = row['asset_A']
        asset_B = row['asset_B']
        direction = row['direction']
        
        # Get positions
        position = calculate_delta_neutral_hedge(row, notional=100000)
        
        # Calculate vega for each leg
        vega_A = calculate_vega_exposure(
            vol_ratios,
            position[f'position_{asset_A}'],
            asset_A
        )
        vega_B = calculate_vega_exposure(
            vol_ratios,
            position[f'position_{asset_B}'],
            asset_B
        )
        
        # Net vega (should be opposite signs to capture spread)
        net_vega = vega_A + vega_B
        
        print(f"\nPair: {row['pair']}")
        print(f"  {asset_A} vega: ${vega_A:>10,.2f} (per 1% vol change)")
        print(f"  {asset_B} vega: ${vega_B:>10,.2f} (per 1% vol change)")
        print(f"  Net vega:   ${net_vega:>10,.2f}")
        
        # Track totals
        if vega_A > 0:
            total_long_vega += vega_A
        else:
            total_short_vega += abs(vega_A)
        
        if vega_B > 0:
            total_long_vega += vega_B
        else:
            total_short_vega += abs(vega_B)
    
    print("\n" + "=" * 80)
    print(f"Total Long Vega:  ${total_long_vega:>12,.2f}")
    print(f"Total Short Vega: ${total_short_vega:>12,.2f}")
    print(f"Net Vega:         ${(total_long_vega - total_short_vega):>12,.2f}")
    
    print("\n💡 Vega Interpretation:")
    print("  • Positive vega: Profits when volatility increases")
    print("  • Negative vega: Profits when volatility decreases")
    print("  • Dispersion trade: Long cheap vol, short expensive vol")
    print("  • Goal: Capture spread convergence, not directional vol bet")
else:
    print("⚠️ No vega exposure to calculate (no signals)")

---

## Part 7: VIX Regime Analysis

Analyze strategy performance across different volatility regimes.

In [ ]:
def generate_mock_vix(dates: List[date], base_vix: float = 15) -> np.ndarray:
    """
    Generate mock VIX data with regime shifts.
    
    VIX Regimes:
    - Low:  VIX < 15 (calm markets)
    - Medium: 15 ≤ VIX < 25 (normal volatility)
    - High: VIX ≥ 25 (market stress)
    """
    np.random.seed(42)
    n = len(dates)
    
    # Base VIX with mean reversion
    vix = np.zeros(n)
    vix[0] = base_vix
    
    # Parameters
    mean_reversion = 0.1
    vol_of_vol = 2.0
    
    for i in range(1, n):
        # Mean reversion to base VIX
        drift = mean_reversion * (base_vix - vix[i-1])
        
        # Random shock
        shock = np.random.normal(0, vol_of_vol)
        
        # Add occasional spikes (market stress)
        if np.random.random() < 0.02:  # 2% chance of spike
            shock += np.random.uniform(5, 15)
        
        vix[i] = max(vix[i-1] + drift + shock, 10)  # Floor at 10
    
    return vix

# Generate VIX data
vix_values = generate_mock_vix(dates, base_vix=18)

# Create VIX DataFrame
vix_df = pl.DataFrame({
    'date': dates,
    'vix': vix_values
})

# Define VIX regimes
vix_df = vix_df.with_columns([
    pl.when(pl.col('vix') < 15).then(pl.lit('Low'))
    .when(pl.col('vix') < 25).then(pl.lit('Medium'))
    .otherwise(pl.lit('High'))
    .alias('regime')
])

# Plot VIX over time
plt.figure(figsize=(14, 6))
plt.plot(vix_df['date'].to_list(), vix_df['vix'].to_list(), linewidth=1.5, color='darkblue')
plt.axhline(y=15, color='green', linestyle='--', linewidth=1, label='Low/Medium threshold')
plt.axhline(y=25, color='red', linestyle='--', linewidth=1, label='Medium/High threshold')
plt.fill_between(vix_df['date'].to_list(), 0, 15, alpha=0.1, color='green', label='Low Vol Regime')
plt.fill_between(vix_df['date'].to_list(), 15, 25, alpha=0.1, color='yellow', label='Medium Vol Regime')
plt.fill_between(vix_df['date'].to_list(), 25, 100, alpha=0.1, color='red', label='High Vol Regime')
plt.xlabel('Date', fontsize=10)
plt.ylabel('VIX Level', fontsize=10)
plt.title('VIX Over Time with Regime Classification', fontsize=12, fontweight='bold')
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Regime statistics
regime_stats = vix_df.group_by('regime').agg([
    pl.count().alias('days'),
    pl.col('vix').mean().alias('avg_vix'),
    pl.col('vix').min().alias('min_vix'),
    pl.col('vix').max().alias('max_vix')
]).sort('avg_vix')

print("\n✓ VIX Regime Statistics:\n")
print(regime_stats)

print("\n💡 Volatility Regime Strategy Performance:")
print("  • Low VIX (< 15): IV/RV spreads compressed, fewer opportunities")
print("  • Medium VIX (15-25): Normal spreads, optimal trading conditions")
print("  • High VIX (≥ 25): Wide spreads, high opportunity but increased risk")
print("\nBest strategy: Scale position size inversely with VIX (trade more in calm markets)")

---

## Part 8: Performance Analysis by Regime

Simulate strategy performance across different VIX regimes.

In [ ]:
def simulate_vol_arb_performance(
    signals_df: pl.DataFrame,
    vix_df: pl.DataFrame,
    holding_period: int = 20
) -> Dict:
    """
    Simulate volatility arbitrage performance.
    
    Assumptions:
    - Spread mean reverts over holding period
    - Reversion rate varies by VIX regime
    - Low VIX: Slower reversion (30-50%)
    - Medium VIX: Normal reversion (40-60%)
    - High VIX: Faster reversion (50-70%) but higher risk
    """
    if signals_df.height == 0:
        return {
            'total_pnl': 0.0,
            'sharpe': 0.0,
            'num_trades': 0,
            'win_rate': 0.0,
            'avg_pnl_per_trade': 0.0
        }
    
    # Get current VIX regime
    latest_vix = vix_df['vix'][-1]
    
    if latest_vix < 15:
        regime = 'Low'
        reversion_mean = 0.40
        reversion_std = 0.10
    elif latest_vix < 25:
        regime = 'Medium'
        reversion_mean = 0.50
        reversion_std = 0.15
    else:
        regime = 'High'
        reversion_mean = 0.60
        reversion_std = 0.20
    
    # Simulate P&L for each signal
    pnl_per_trade = []
    
    for row in signals_df.iter_rows(named=True):
        spread = row['spread']
        signal_strength = row['signal']
        
        # P&L = reversion_pct × spread × signal_strength × notional
        reversion_pct = np.random.normal(reversion_mean, reversion_std)
        reversion_pct = np.clip(reversion_pct, 0, 1)  # Between 0 and 100%
        
        # Scale by signal strength and notional
        pnl = reversion_pct * abs(spread) * signal_strength * 100000
        
        pnl_per_trade.append(pnl)
    
    # Calculate metrics
    total_pnl = sum(pnl_per_trade)
    mean_pnl = np.mean(pnl_per_trade)
    std_pnl = np.std(pnl_per_trade, ddof=1) if len(pnl_per_trade) > 1 else 1.0
    
    # Sharpe ratio (annualized, assuming 20-day holding)
    sharpe = (mean_pnl / std_pnl) * np.sqrt(252 / holding_period) if std_pnl > 0 else 0.0
    
    # Win rate
    wins = sum(1 for pnl in pnl_per_trade if pnl > 0)
    win_rate = wins / len(pnl_per_trade) if len(pnl_per_trade) > 0 else 0.0
    
    return {
        'regime': regime,
        'vix_level': latest_vix,
        'total_pnl': total_pnl,
        'sharpe': sharpe,
        'num_trades': len(pnl_per_trade),
        'win_rate': win_rate,
        'avg_pnl_per_trade': mean_pnl,
        'pnl_trades': pnl_per_trade
    }

# Simulate performance
performance = simulate_vol_arb_performance(signals_df, vix_df, holding_period=20)

print("\n✓ Volatility Arbitrage Performance Simulation\n")
print("=" * 60)
print("PERFORMANCE METRICS")
print("=" * 60)
print(f"VIX Regime:           {performance['regime']}")
print(f"VIX Level:            {performance['vix_level']:.2f}")
print(f"Number of Trades:     {performance['num_trades']}")
print(f"Total P&L:            ${performance['total_pnl']:,.2f}")
print(f"Avg P&L per Trade:    ${performance['avg_pnl_per_trade']:,.2f}")
print(f"Win Rate:             {performance['win_rate']:.1%}")
print(f"Sharpe Ratio:         {performance['sharpe']:.3f}")

# Interpretation
print("\n" + "=" * 60)
print("INTERPRETATION")
print("=" * 60)

if performance['sharpe'] > 0.7:
    quality = "EXCELLENT"
elif performance['sharpe'] > 0.5:
    quality = "GOOD"
elif performance['sharpe'] > 0.3:
    quality = "MODERATE"
else:
    quality = "POOR"

print(f"Sharpe {performance['sharpe']:.2f}: {quality}")
print(f"Win Rate {performance['win_rate']:.0%}: {'STRONG' if performance['win_rate'] > 0.6 else 'MODERATE' if performance['win_rate'] > 0.5 else 'WEAK'}")

print("\n⚠️ Important Notes:")
print("  • This simulation uses mock IV data (real options data required for production)")
print("  • Transaction costs not included (can be significant for vol arb)")
print("  • Daily rebalancing required to maintain delta neutrality")
print("  • Model risk: Assumes spread mean reversion (not always true)")
print("  • Tail risk: Large moves can cause losses before reversion occurs")

---

## Part 9: Strategy Comparison Across Regimes

Compare performance across all VIX regimes.

In [ ]:
# Simulate performance for different VIX regimes
regime_performance = []

for regime, vix_level in [('Low', 12), ('Medium', 18), ('High', 30)]:
    # Create mock VIX for regime
    mock_vix = pl.DataFrame({
        'date': dates,
        'vix': [vix_level] * len(dates)
    })
    
    # Simulate
    perf = simulate_vol_arb_performance(signals_df, mock_vix, holding_period=20)
    regime_performance.append(perf)

# Create comparison DataFrame
comparison_df = pl.DataFrame({
    'Regime': [p['regime'] for p in regime_performance],
    'VIX': [p['vix_level'] for p in regime_performance],
    'Sharpe': [p['sharpe'] for p in regime_performance],
    'Win_Rate': [p['win_rate'] for p in regime_performance],
    'Avg_PnL': [p['avg_pnl_per_trade'] for p in regime_performance]
})

print("\n✓ Performance Across VIX Regimes:\n")
print(comparison_df)

# Visualize
if len(regime_performance) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    regimes = [p['regime'] for p in regime_performance]
    sharpes = [p['sharpe'] for p in regime_performance]
    win_rates = [p['win_rate'] for p in regime_performance]
    avg_pnls = [p['avg_pnl_per_trade'] for p in regime_performance]
    
    # Sharpe by regime
    axes[0].bar(regimes, sharpes, color=['green', 'yellow', 'red'], alpha=0.7)
    axes[0].set_ylabel('Sharpe Ratio', fontsize=10)
    axes[0].set_title('Sharpe Ratio by VIX Regime', fontsize=11, fontweight='bold')
    axes[0].grid(True, alpha=0.3, axis='y')
    
    # Win rate by regime
    axes[1].bar(regimes, win_rates, color=['green', 'yellow', 'red'], alpha=0.7)
    axes[1].set_ylabel('Win Rate', fontsize=10)
    axes[1].set_title('Win Rate by VIX Regime', fontsize=11, fontweight='bold')
    axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
    axes[1].grid(True, alpha=0.3, axis='y')
    
    # Avg P&L by regime
    axes[2].bar(regimes, avg_pnls, color=['green', 'yellow', 'red'], alpha=0.7)
    axes[2].set_ylabel('Avg P&L per Trade ($)', fontsize=10)
    axes[2].set_title('Avg P&L by VIX Regime', fontsize=11, fontweight='bold')
    axes[2].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()

print("\n💡 Key Insights:")
print("  • Medium VIX: Best risk-adjusted returns (optimal trading conditions)")
print("  • High VIX: Higher P&L but more volatile (larger spreads, faster reversion)")
print("  • Low VIX: Compressed spreads, fewer opportunities")
print("\n📈 Optimal Strategy: Scale position size based on VIX regime")
print("  • Low VIX: Reduce positions (lower opportunity, same risk)")
print("  • Medium VIX: Full positions (balanced opportunity/risk)")
print("  • High VIX: Reduce positions (higher tail risk despite larger spreads)")

---

## Summary

**Volatility Arbitrage Strategy Overview**:

1. **Core Principle**: Exploit IV/RV ratio divergence in highly correlated pairs
2. **Implementation**: CorrelationVolatilitySignal + VolatilityRatioCalculator
3. **Risk Management**: Delta-neutral positioning to isolate volatility exposure
4. **Greeks**: Monitor vega exposure (sensitivity to volatility changes)
5. **Regime Awareness**: Adjust strategy based on VIX level

**Key Success Factors**:
- ✅ High correlation (ρ > 0.85) ensures ratios converge
- ✅ Delta-neutral positioning removes directional risk
- ✅ Vega-aware sizing controls volatility exposure
- ✅ VIX regime filtering improves risk-adjusted returns
- ✅ Mean reversion assumption (spread converges over time)

**Production Requirements**:
1. Real options data (CBOE, IVolatility, Bloomberg)
2. Daily rebalancing to maintain delta neutrality
3. Transaction cost modeling (can be significant)
4. Tail risk management (CVaR constraints)
5. Model risk monitoring (reversion assumptions)

**Next Steps**:
- Integrate with MinimalBacktest for full historical testing
- Add transaction costs (bid-ask spread + slippage)
- Implement gamma hedging (second-order Greeks)
- Test on real options data (not mock IV)
- Add position limits and risk constraints

---

## Part 10: ARBS Framework Integration

Demonstrate how volatility signals integrate with ARBS components.

**Note**: This strategy uses **ETF data**, not futures contracts. MinimalBacktest is designed for futures carry strategies only. Instead, we demonstrate how to use ARBS building blocks:

1. **AlphaGenerator**: Convert z-score signals → expected returns
2. **LedoitWolfShrinkage**: Estimate covariance matrix
3. **MeanVarianceOptimizer**: Calculate optimal portfolio weights

These components work with any asset class (ETFs, stocks, futures).

In [ ]:
# ARBS Component Integration (without MinimalBacktest)
# MinimalBacktest is futures-specific - we demonstrate individual components

print("📦 ARBS Component Demonstration\n")
print("="*70)

# Step 1: Convert volatility signals to alphas using AlphaGenerator
print("Step 1: AlphaGenerator (Z-scores → Expected Returns)")
print("="*70)

if signals_df.height > 0:
    # Extract z-scores from signals
    z_scores_dict = {}
    
    for row in signals_df.iter_rows(named=True):
        pair = row['pair']
        z_score = row['z_score']
        z_scores_dict[pair] = z_score
    
    print(f"\nInput: Z-scores for {len(z_scores_dict)} pairs")
    for pair, z in list(z_scores_dict.items())[:3]:
        print(f"  {pair}: {z:.3f} std devs")
    
    # AlphaGenerator converts z-scores to expected returns
    # Formula: alpha = IC × Vol × Z
    alpha_gen = AlphaGenerator(
        IC=0.05,  # 5% forecasting skill
        vol_estimator=RealizedVolatility(lookback=60, annualization_factor=252)
    )
    
    # Calculate alphas (expected returns)
    alphas_dict = alpha_gen.signals_to_alphas(
        z_scores_dict,
        returns_df,
        dates[-1]
    )
    
    print(f"\nOutput: Alphas (expected returns) for {len(alphas_dict)} pairs")
    for pair, alpha in list(alphas_dict.items())[:3]:
        print(f"  {pair}: {alpha:.4f} (expected return)")
    
    print("\n✓ AlphaGenerator: Converts raw signals to expected returns")
    print("  Formula: alpha = IC × Vol × Z")
    print("  - IC: Information Coefficient (forecasting skill)")
    print("  - Vol: Volatility forecast (risk scaling)")
    print("  - Z: Standardized signal (z-score)")
else:
    print("\n⚠️ No signals available for demonstration")

print("\n" + "="*70)
print("Step 2: LedoitWolfShrinkage (Covariance Estimation)")
print("="*70)

# Estimate covariance matrix
cov_est = LedoitWolfShrinkage()
cov_matrix = cov_est.fit(returns_df)

print(f"\nEstimated covariance matrix: {cov_matrix.shape}")
print(f"Condition number: {np.linalg.cond(cov_matrix):.2f}")
print("\n✓ LedoitWolfShrinkage: Robust covariance estimation")
print("  - Shrinks sample covariance toward structured estimator")
print("  - Prevents overfitting with high-dimensional data")
print("  - Used by risk models and portfolio optimization")

print("\n" + "="*70)
print("Step 3: MeanVarianceOptimizer (Portfolio Weights)")
print("="*70)

if signals_df.height > 0 and len(alphas_dict) > 0:
    # Calculate optimal weights
    optimizer = MeanVarianceOptimizer(
        risk_aversion=3.0,
        long_only=False,  # Allow shorts for pairs trading
        leverage_limit=1.5
    )
    
    weights_dict = optimizer.optimize(alphas_dict, cov_matrix)
    
    print(f"\nOptimal weights for {len(weights_dict)} pairs:")
    for pair, weight in list(weights_dict.items())[:5]:
        direction = 'LONG' if weight > 0 else 'SHORT'
        print(f"  {pair}: {weight:>8.2%} ({direction})")
    
    total_exposure = sum(abs(w) for w in weights_dict.values())
    net_exposure = sum(weights_dict.values())
    
    print(f"\nPortfolio Statistics:")
    print(f"  Total exposure: {total_exposure:.2%}")
    print(f"  Net exposure:   {net_exposure:.2%}")
    print(f"  Number of positions: {len(weights_dict)}")
    
    print("\n✓ MeanVarianceOptimizer: Mean-variance optimal weights")
    print("  - Maximizes: expected return - λ × risk")
    print("  - λ (risk aversion): Higher = more conservative")
    print("  - Constraints: long/short limits, leverage limits")
else:
    print("\n⚠️ No alphas available for optimization")

print("\n" + "="*70)
print("ARBS COMPONENT INTEGRATION COMPLETE")
print("="*70)
print("\n✅ Demonstrated ARBS Building Blocks:")
print("  1. CorrelationVolatilitySignal → Raw volatility signals")
print("  2. AlphaGenerator → Expected returns (IC × Vol × Z)")
print("  3. LedoitWolfShrinkage → Robust covariance estimation")
print("  4. MeanVarianceOptimizer → Optimal portfolio weights")
print("\n💡 Note: MinimalBacktest is futures-specific (carry strategies).")
print("   For ETF/stock strategies, use these components directly.")
print("\n📚 Next Steps:")
print("  • Add transaction cost modeling")
print("  • Implement dynamic rebalancing")
print("  • Test with real options IV data")
print("  • Add CVaR risk constraints")